# PyTorch: Interactive Tour

If NumPy already feels comfortable, PyTorch is 90% the same API on a type
called `Tensor` instead of `ndarray`, plus two things NumPy doesn't have:
**GPU support** and **autograd** (automatic differentiation, how models get
trained). This is what every mech interp notebook you've used runs on.


```python
import torch
print(torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
```


In [ ]:
# type it yourself, then run

## Tensors: same shape/dtype rules as NumPy, plus `device`

A tensor lives on exactly one device (`"cpu"` or `"cuda"`) at a time. This
is the single most common bug source for newcomers: you cannot combine or
compare a CPU tensor and a GPU tensor directly, and you cannot hand a GPU
tensor to a library (like `numpy` or `plotly`) that expects CPU memory.
That's why every plotting call in the mech interp notebooks ends with
`.cpu()` before it hits `plotly`.


```python
t = torch.tensor([1.0, 2.0, 3.0])
print(t.shape, t.dtype, t.device)

t_gpu = t.to(device)
print(t_gpu.device)

# interop with numpy, only works when the tensor is on CPU
import numpy as np
print(t.numpy())
print(torch.from_numpy(np.array([4.0, 5.0])))
```


In [ ]:
# type it yourself, then run



```python
try:
    t_gpu + t  # GPU tensor + CPU tensor
except RuntimeError as e:
    print("As expected, this fails:", e)
```


In [ ]:
# type it yourself, then run

## Indexing, slicing, broadcasting: identical to NumPy

Everything from the numpy notebook (`[start:stop:step]`, boolean masks,
broadcasting rules) works exactly the same way on tensors. What's new below
is reshaping vocabulary and autograd.


```python
m = torch.arange(12).reshape(3, 4)
print(m)
print(m[m % 2 == 0])
print((torch.ones(3, 1) + torch.ones(1, 4)).shape)  # broadcasts to (3, 4)
```


In [ ]:
# type it yourself, then run

## `view` vs `reshape`, and `squeeze`/`unsqueeze`

- `.reshape()` always works, like NumPy's.
- `.view()` is a faster reshape that *shares memory* with the original
  tensor, but only works when the data is laid out contiguously in memory.
  If you hit `RuntimeError: view size is not compatible...`, switch to
  `.reshape()`, which handles the non-contiguous case by copying.
- `.unsqueeze(dim)` inserts a size-1 dimension (like NumPy's `newaxis`).
  `.squeeze()` removes all size-1 dimensions.


```python
v = torch.arange(6)
print(v.view(2, 3))
print(v.unsqueeze(0).shape)   # (6,) -> (1, 6)
print(v.unsqueeze(0).squeeze().shape)  # back to (6,)

# the .T you'd write in numpy often needs .transpose(dim0, dim1) in torch
# for tensors with more than 2 dimensions
x = torch.zeros(2, 3, 4)
print(x.transpose(0, 1).shape)  # swap dims 0 and 1 -> (3, 2, 4)
```


In [ ]:
# type it yourself, then run

## Autograd: how gradients get computed

Every mech interp notebook you ran started with `torch.set_grad_enabled(False)`
because we only ever did inference, never training. Here's what that flag
turns off. Set `requires_grad=True` on a tensor, do some computation, call
`.backward()` on a scalar result, and PyTorch fills in `.grad` on every
tensor that fed into it. That's the whole mechanism behind training a model
with gradient descent (and PPO, which you already know, is built on this
same primitive).


```python
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1   # y = x^2 + 2x + 1, dy/dx = 2x + 2

y.backward()
print("x.grad:", x.grad.item())        # should be 2*3 + 2 = 8
print("matches by hand:", 2 * 3.0 + 2)
```


In [ ]:
# type it yourself, then run

## Turning gradient tracking off

`torch.no_grad()` (or `torch.set_grad_enabled(False)`) is exactly what
every mech interp notebook opens with, and here's why. We don't need
gradients for inference, and tracking them wastes memory.


```python
with torch.no_grad():
    z = x ** 2
    print(z.requires_grad)   # False inside the block
```


In [ ]:
# type it yourself, then run

## A minimal `nn.Module`

Every layer in a real model (including every `HookedTransformer` block you
inspected) is built from this same pattern: a class holding learnable
parameters, with a `forward` method defining what happens to an input.


```python
import torch.nn as nn

linear = nn.Linear(in_features=4, out_features=2)
print("weight shape:", linear.weight.shape)   # (out_features, in_features)
print("bias shape:  ", linear.bias.shape)

x = torch.randn(5, 4)   # batch of 5 inputs, each of dimension 4
out = linear(x)         # calls linear.forward(x) under the hood
print("output shape:", out.shape)   # (5, 2)
```


In [ ]:
# type it yourself, then run

## `topk`, and recovering `(layer, head)` from a flat index

`.topk(k)` returns a named tuple with `.values` and `.indices`, the k
largest elements and where they came from. When you have a 2D grid like a
`[n_layers, n_heads]` score tensor and want the single best `(layer, head)`
pair, the standard move is to flatten to 1D, call `.argmax()` or `.topk()`
on that, then convert the flat index back to 2D coordinates with
`divmod(flat_idx, n_heads)`. This exact idiom shows up constantly in mech
interp code for figuring out which head did something.


```python
scores = torch.tensor([[0.1, 0.9, 0.3], [0.7, 0.2, 0.4]])  # [n_layers=2, n_heads=3]
flat_idx = scores.flatten().argmax().item()
layer, head = divmod(flat_idx, scores.shape[1])
print(f"best: layer {layer}, head {head}, score {scores[layer, head]}")

top3 = scores.flatten().topk(3)
print(top3.values, top3.indices)
```


In [ ]:
# type it yourself, then run

## `.diagonal()` on a batched tensor

`.diagonal(offset=k, dim1=, dim2=)` pulls out a diagonal (optionally shifted
by `offset`) from two of a tensor's dimensions, leaving the rest alone. This
is the mechanic behind scoring induction heads: an attention pattern has
shape `[batch, head, dest_pos, src_pos]`, and "attends to the position right
after the token it's copying" is exactly the diagonal at `offset=-seq_len+1`
between the last two dims.


```python
pattern = torch.rand(2, 4, 6, 6)  # [batch, head, dest_pos, src_pos]
diag = pattern.diagonal(offset=-2, dim1=-2, dim2=-1)
print(diag.shape)  # the diagonal becomes the last dim: [batch, head, diag_len]
```


In [ ]:
# type it yourself, then run

## Reducing over more than one axis at once

`dim` accepts a tuple, collapsing several axes in a single call. For
example, averaging the diagonal above over both the batch dimension and its
own length gives one induction score per head.


```python
per_head_score = diag.mean(dim=(0, -1))  # collapses batch AND diag_len -> [head]
print(per_head_score.shape)
```


In [ ]:
# type it yourself, then run

## Editing a tensor in place with slice assignment

Reading a slice (`t[:, 5, :]`) gives you a view or copy; *assigning* to a
slice (`t[:, 5, :] = other`) mutates the original tensor's data directly.
This is the mechanic behind activation patching: a hook function receives an
activation, overwrites one position's worth of values with a value cached
from a different forward pass, and returns it.


```python
resid = torch.zeros(2, 5, 3)    # [batch, pos, d_model]
replacement = torch.ones(2, 3)  # [batch, d_model]
resid[:, 2, :] = replacement
print(resid[:, 2, :])
print(resid[:, 0, :])  # untouched
```


In [ ]:
# type it yourself, then run

## `functools.partial` for fixed-signature callbacks

Callback-style APIs (hooks, event handlers) usually require your function to
have one exact signature. You can't just tack on extra parameters.
`functools.partial` pre-binds extra arguments to a function, producing a new
function that still matches the required signature.


```python
from functools import partial

def scale_and_add(x, hook_name, factor):
    return x * factor + len(hook_name)

bound = partial(scale_and_add, hook_name="blocks.0.attn.hook_pattern", factor=2.0)
print(bound(torch.tensor(3.0)))  # calls scale_and_add(3.0, hook_name=..., factor=2.0)
```


In [ ]:
# type it yourself, then run

## Boolean-mask assignment

You've already read with a mask (`t[mask]`); the same mask works on the left
side of `=` to overwrite exactly those positions, leaving everything else
untouched.


```python
t = torch.arange(6.0)
t[t > 3] = 0.0
print(t)
```


In [ ]:
# type it yourself, then run

## `.tolist()`, and a habit worth adopting

`.tolist()` converts a tensor to plain nested Python lists or numbers. This
is useful when you need token ids as a list to build strings or labels from,
rather than a tensor.

Also: every mech interp notebook ahead comments each tensor's shape inline,
naming the axes (`# [batch, head, dest_pos, src_pos]`), right after
computing it. It looks trivial but it's the fastest way to catch a wrong
`dim=` argument before it silently produces a wrong-but-plausible-shaped
result. Get in the habit here, not later.


```python
tokens = torch.tensor([[15496, 995]])  # [batch, pos]
print(tokens[0].tolist())
```


In [ ]:
# type it yourself, then run

## Checkpoint: write a few lines yourself

Given `scores` below (a `[n_layers=4, n_heads=3]` grid of fake induction
scores), compute `top3`: a list of the 3 `(layer, head)` tuples with the
highest scores, ordered from highest to lowest. Use `.flatten()`, `.topk()`,
and `divmod`.

In [ ]:
scores = torch.tensor([
    [0.12, 0.88, 0.31],
    [0.65, 0.20, 0.42],
    [0.05, 0.10, 0.97],
    [0.55, 0.60, 0.15],
])

top3 = None  # YOUR CODE HERE

In [ ]:
# self-check
assert top3 is not None, "fill in `top3` above"
assert top3 == [(2, 2), (0, 1), (1, 0)], f"got {top3}"
print("Checkpoint passed")

## Cheat sheet

| Want to... | Call |
|---|---|
| Make a tensor | `torch.tensor(data)`, `torch.zeros(shape)`, `torch.randn(shape)`, `torch.arange(...)` |
| Move to/from GPU | `t.to(device)`, `t.cuda()`, `t.cpu()` |
| Tensor <-> NumPy | `t.numpy()` (CPU only), `torch.from_numpy(arr)` |
| Reshape (safe) | `t.reshape(shape)` |
| Reshape (fast, needs contiguous) | `t.view(shape)` |
| Add/remove a size-1 dim | `t.unsqueeze(dim)`, `t.squeeze()` |
| Swap two axes | `t.transpose(dim0, dim1)` |
| Track gradients | `t = torch.tensor(x, requires_grad=True)` |
| Compute gradients | `loss.backward()`, then read `.grad` |
| Disable gradient tracking | `torch.no_grad()` / `torch.set_grad_enabled(False)` |
| A learnable layer | `nn.Linear(in_features, out_features)` and subclasses of `nn.Module` |
| Top-k values/indices | `t.topk(k)` -> `.values`, `.indices` |
| Flat index -> multi-dim coords | `divmod(flat_idx, size_of_last_dim)` |
| Extract a (shifted) diagonal | `t.diagonal(offset=k, dim1=, dim2=)` |
| Reduce over multiple axes at once | `t.mean(dim=(0, -1))` |
| Edit a slice in place | `t[:, i, :] = other` |
| Bind extra args onto a fixed-signature callback | `functools.partial(fn, extra=val)` |
| Assign into masked positions | `t[mask] = value` |
| Tensor -> plain Python list | `t.tolist()` |